# Retail Decision Intelligence - Data Audit

Exploratory audit of two source datasets:
1. `retail_sales_ml_apl.csv` - Sales transactions
2. `retail_inventory_ml_apl.csv` - Inventory snapshots

**Rule**: Raw data is immutable. No modifications, cleaning, or transformations.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("../data/raw")
sales_path = RAW_DIR / "retail_sales_ml_apl.csv"
inv_path = RAW_DIR / "retail_inventory_ml_apl.csv"

print(f"Sales file size: {sales_path.stat().st_size / 1e6:.2f} MB")
print(f"Inventory file size: {inv_path.stat().st_size / 1e6:.2f} MB")

In [ ]:
# Load data with appropriate dtypes
sales = pd.read_csv(sales_path, low_memory=False)
inv = pd.read_csv(inv_path, low_memory=False)

print(f"Sales shape: {sales.shape}")
print(f"Inventory shape: {inv.shape}")

print("\nSales columns:")
print(sales.columns.tolist())
print("\nInventory columns:")
print(inv.columns.tolist())

In [ ]:
# Basic info
print("=== SALES ===")
sales.info()
print("\n=== INVENTORY ===")
inv.info()

In [ ]:
# Date ranges
sales['Transaction Date'] = pd.to_datetime(sales['Transaction Date'])
inv['Start Date'] = pd.to_datetime(inv['Start Date'])
inv['End Date'] = pd.to_datetime(inv['End Date'])

print(f"Sales date range: {sales['Transaction Date'].min()} to {sales['Transaction Date'].max()}")
print(f"Inventory Start date range: {inv['Start Date'].min()} to {inv['Start Date'].max()}")
print(f"Inventory End date range: {inv['End Date'].min()} to {inv['End Date'].max()}")

In [ ]:
# Unique counts
print("=== UNIQUE COUNTS ===")
print(f"Unique stores (sales): {sales['Store'].nunique()}")
print(f"Unique stores (inv): {inv['Store'].nunique()}")
print(f"Unique products/SKUs (sales): {sales['Product No'].nunique()}")
print(f"Unique products/SKUs (inv): {inv['Product No'].nunique()}")
print(f"Unique suppliers (sales): {sales['Supplier'].nunique()}")
print(f"Unique suppliers (inv): {inv['Supplier'].nunique()}")
print(f"Product Divisions (sales): {sales['Product Division'].nunique()}")
print(f"Product Categories (sales): {sales['Product Category'].nunique()}")
print(f"Product Subcategories (sales): {sales['Product Subcategory'].nunique()}")
print(f"Product Segments (sales): {sales['Product Segment'].nunique()}")

In [ ]:
# Duplicate records
print(f"Sales duplicate rows: {sales.duplicated().sum()}")
print(f"Inventory duplicate rows: {inv.duplicated().sum()}")

# Check for duplicate keys
sales_key_dupes = sales.duplicated(subset=['Transaction Date', 'Store', 'Product No']).sum()
inv_key_dupes = inv.duplicated(subset=['Start Date', 'End Date', 'Store', 'Product No']).sum()
print(f"Sales duplicate (date, store, product): {sales_key_dupes}")
print(f"Inventory duplicate (start, end, store, product): {inv_key_dupes}")

In [ ]:
# Missing values
print("=== SALES MISSING ===")
print(sales.isnull().sum())
print("\n=== INVENTORY MISSING ===")
print(inv.isnull().sum())

In [ ]:
# Numerical field analysis - Sales
num_cols_sales = ['Qty Sold', 'Sales Amount', 'Cogs', 'Number of Transactions']
for col in num_cols_sales:
    s = sales[col]
    print(f"\n{col}:")
    print(f"  Min: {s.min()}, Max: {s.max()}, Mean: {s.mean():.2f}, Median: {s.median()}")
    print(f"  Zeros: {(s == 0).sum()}, Negatives: {(s < 0).sum()}")
    print(f"  Missing: {s.isnull().sum()}")

In [ ]:
# Numerical field analysis - Inventory
num_cols_inv = ['Qty on hand', 'Stocks Selling Amount', 'Cost of Stocks', 'Stock Unit Selling Price', 'Stock Unit Cost Price']
for col in num_cols_inv:
    s = inv[col]
    print(f"\n{col}:")
    print(f"  Min: {s.min()}, Max: {s.max()}, Mean: {s.mean():.2f}, Median: {s.median()}")
    print(f"  Zeros: {(s == 0).sum()}, Negatives: {(s < 0).sum()}")
    print(f"  Missing: {s.isnull().sum()}")

In [ ]:
# Sales Type distribution
print("Sales Type:")
print(sales['Sales Type'].value_counts())
print(f"\nIs Return:")
print(sales['Is Return'].value_counts())
print(f"\nReason of Return (non-null):")
print(sales['Reason of Return'].dropna().value_counts())

In [ ]:
# Returns analysis
returns = sales[sales['Is Return'] == 1]
print(f"Return transactions: {len(returns)}")
if len(returns) > 0:
    print(f"Return Qty Sold stats:")
    print(returns['Qty Sold'].describe())
    print(f"Return Sales Amount stats:")
    print(returns['Sales Amount'].describe())

In [ ]:
# Stock Status distribution
print("Stock Status:")
print(inv['Stock Status'].value_counts())
print(f"\nStore Type:")
print(inv['Store Type'].value_counts())
print(f"\nSales Channel:")
print(inv['Sales Channel'].value_counts())

In [ ]:
# Inventory duration analysis
inv['Duration Days'] = (inv['End Date'] - inv['Start Date']).dt.days
print(f"Inventory duration stats:")
print(inv['Duration Days'].describe())

# Check for same-day start/end (potential stockout snapshots)
same_day = inv[inv['Duration Days'] == 0]
print(f"\nSame-day start/end records: {len(same_day)} ({len(same_day)/len(inv)*100:.1f}%)")

In [ ]:
# Can inventory be reconstructed at daily product-store level?
# Check if inventory records are continuous or snapshots
sample_product = inv['Product No'].iloc[0]
sample_store = inv['Store'].iloc[0]
subset = inv[(inv['Product No'] == sample_product) & (inv['Store'] == sample_store)].sort_values('Start Date')
print(f"Sample product-store inventory records: {len(subset)}")
print(subset[['Start Date', 'End Date', 'Qty on hand', 'Stock Status']].head(10))

# Check for gaps
subset['Gap Days'] = subset['Start Date'].diff().dt.days
print(f"\nGaps between records (days):")
print(subset['Gap Days'].value_counts().head())

In [ ]:
# Sales-Inventory relationship
# Check overlap in products and stores
sales_products = set(sales['Product No'].unique())
inv_products = set(inv['Product No'].unique())
sales_stores = set(sales['Store'].unique())
inv_stores = set(inv['Store'].unique())

print(f"Products in sales only: {len(sales_products - inv_products)}")
print(f"Products in inventory only: {len(inv_products - sales_products)}")
print(f"Products in both: {len(sales_products & inv_products)}")
print(f"\nStores in sales only: {len(sales_stores - inv_stores)}")
print(f"Stores in inventory only: {len(inv_stores - sales_stores)}")
print(f"Stores in both: {len(sales_stores & inv_stores)}")

In [ ]:
# Price, cost, discount, promotion, markdown fields
print("=== PRICE/COST ANALYSIS ===")
print(f"Sales: Unit price implied = Sales Amount / Qty Sold")
sales['Unit Price'] = sales['Sales Amount'] / sales['Qty Sold']
sales['Unit Cost'] = sales['Cogs'] / sales['Qty Sold']
sales['Margin'] = sales['Unit Price'] - sales['Unit Cost']
sales['Margin Pct'] = sales['Margin'] / sales['Unit Price']

print(f"Sales Unit Price: min={sales['Unit Price'].min():.2f}, max={sales['Unit Price'].max():.2f}, mean={sales['Unit Price'].mean():.2f}")
print(f"Sales Unit Cost: min={sales['Unit Cost'].min():.2f}, max={sales['Unit Cost'].max():.2f}, mean={sales['Unit Cost'].mean():.2f}")
print(f"Sales Margin Pct: min={sales['Margin Pct'].min():.4f}, max={sales['Margin Pct'].max():.4f}, mean={sales['Margin Pct'].mean():.4f}")

print(f"\nInventory Unit Selling Price: min={inv['Stock Unit Selling Price'].min():.2f}, max={inv['Stock Unit Selling Price'].max():.2f}, mean={inv['Stock Unit Selling Price'].mean():.2f}")
print(f"Inventory Unit Cost Price: min={inv['Stock Unit Cost Price'].min():.2f}, max={inv['Stock Unit Cost Price'].max():.2f}, mean={inv['Stock Unit Cost Price'].mean():.2f}")

In [ ]:
# Promotion analysis
print("Sales Type breakdown:")
promo_sales = sales[sales['Sales Type'] == 'Promo']
full_price_sales = sales[sales['Sales Type'] == 'Full Price']
print(f"Promo transactions: {len(promo_sales)} ({len(promo_sales)/len(sales)*100:.1f}%)")
print(f"Full Price transactions: {len(full_price_sales)} ({len(full_price_sales)/len(sales)*100:.1f}%)")

print(f"\nPromo avg unit price: {promo_sales['Unit Price'].mean():.2f}")
print(f"Full Price avg unit price: {full_price_sales['Unit Price'].mean():.2f}")

# Check for markdown/discount fields - none explicit, but Sales Type indicates promo

In [ ]:
# Suspicious patterns
print("=== SUSPICIOUS PATTERNS ===")

# 1. Negative or zero sales amount with positive qty
suspect1 = sales[(sales['Sales Amount'] <= 0) & (sales['Qty Sold'] > 0)]
print(f"Zero/negative sales amount with positive qty: {len(suspect1)}")

# 2. Positive sales amount with zero/negative qty
suspect2 = sales[(sales['Sales Amount'] > 0) & (sales['Qty Sold'] <= 0)]
print(f"Positive sales amount with zero/negative qty: {len(suspect2)}")

# 3. COGS > Sales Amount (loss-making transactions)
suspect3 = sales[sales['Cogs'] > sales['Sales Amount']]
print(f"COGS > Sales Amount: {len(suspect3)} ({len(suspect3)/len(sales)*100:.2f}%)")

# 4. Inventory with zero qty on hand but positive selling amount
suspect4 = inv[(inv['Qty on hand'] == 0) & (inv['Stocks Selling Amount'] > 0)]
print(f"Inv zero qty but positive selling amount: {len(suspect4)}")

# 5. Inventory unit price mismatch
inv['Implied Unit Price'] = inv['Stocks Selling Amount'] / inv['Qty on hand']
price_mismatch = inv[abs(inv['Implied Unit Price'] - inv['Stock Unit Selling Price']) > 0.01]
print(f"Inventory unit price mismatch: {len(price_mismatch)}")

# 6. Future dates in sales
future_sales = sales[sales['Transaction Date'] > pd.Timestamp('2025-12-31')]
print(f"Sales with future dates (>2025): {len(future_sales)}")

In [ ]:
# Distribution analysis - Sales Quantity
print("=== SALES QUANTITY DISTRIBUTION ===")
print(sales['Qty Sold'].describe())
print(f"\nValue counts:")
print(sales['Qty Sold'].value_counts().head(20))

In [ ]:
# Distribution analysis - Revenue/Sales Amount
print("=== SALES AMOUNT DISTRIBUTION ===")
print(sales['Sales Amount'].describe())
print(f"\nPercentiles:")
print(sales['Sales Amount'].quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

In [ ]:
# Distribution analysis - Inventory Quantity
print("=== INVENTORY QTY ON HAND DISTRIBUTION ===")
print(inv['Qty on hand'].describe())
print(f"\nValue counts:")
print(inv['Qty on hand'].value_counts().head(20))

In [ ]:
# Stockout analysis
print("=== STOCKOUT ANALYSIS ===")

# Records with zero inventory
zero_inv = inv[inv['Qty on hand'] == 0]
print(f"Zero inventory records: {len(zero_inv)} ({len(zero_inv)/len(inv)*100:.1f}%)")

# Stock Status for zero qty
if len(zero_inv) > 0:
    print(f"Stock Status for zero qty:")
    print(zero_inv['Stock Status'].value_counts())

# Check if 'Stockout' status exists
print(f"\nAll Stock Status values:")
print(inv['Stock Status'].unique())

In [ ]:
# Product hierarchy analysis
print("=== PRODUCT HIERARCHY ===")
hierarchy = sales[['Product Division', 'Product Category', 'Product Subcategory', 'Product Segment']].drop_duplicates()
print(f"Total unique hierarchy combinations: {len(hierarchy)}")

print(f"\nDivisions:")
print(sales['Product Division'].value_counts())

print(f"\nCategories per Division:")
for div in sales['Product Division'].unique():
    cats = sales[sales['Product Division'] == div]['Product Category'].nunique()
    print(f"  {div}: {cats} categories")

In [ ]:
# Temporal coverage - sales by month
sales['YearMonth'] = sales['Transaction Date'].dt.to_period('M')
monthly_sales = sales.groupby('YearMonth').agg(
    transactions=('Transaction Date', 'count'),
    total_qty=('Qty Sold', 'sum'),
    total_revenue=('Sales Amount', 'sum'),
    unique_products=('Product No', 'nunique'),
    unique_stores=('Store', 'nunique')
)
print(monthly_sales)

In [ ]:
# Temporal coverage - inventory by month
inv['Start YearMonth'] = inv['Start Date'].dt.to_period('M')
monthly_inv = inv.groupby('Start YearMonth').agg(
    records=('Start Date', 'count'),
    unique_products=('Product No', 'nunique'),
    unique_stores=('Store', 'nunique'),
    total_qty=('Qty on hand', 'sum')
)
print(monthly_inv)

In [ ]:
# Supplier analysis
print("=== SUPPLIER ANALYSIS ===")
supplier_sales = sales.groupby('Supplier').agg(
    transactions=('Transaction Date', 'count'),
    revenue=('Sales Amount', 'sum'),
    products=('Product No', 'nunique'),
    stores=('Store', 'nunique')
).sort_values('revenue', ascending=False)
print(supplier_sales.head(20))

In [ ]:
# Store analysis
print("=== STORE ANALYSIS ===")
store_sales = sales.groupby('Store').agg(
    transactions=('Transaction Date', 'count'),
    revenue=('Sales Amount', 'sum'),
    products=('Product No', 'nunique')
).sort_values('revenue', ascending=False)
print(store_sales.head(20))

In [ ]:
# Product-level sales summary
print("=== TOP PRODUCTS BY REVENUE ===")
product_sales = sales.groupby(['Product No', 'Product Description', 'Product Division', 'Product Category']).agg(
    total_qty=('Qty Sold', 'sum'),
    total_revenue=('Sales Amount', 'sum'),
    transactions=('Transaction Date', 'count'),
    stores=('Store', 'nunique'),
    avg_unit_price=('Unit Price', 'mean')
).sort_values('total_revenue', ascending=False)
print(product_sales.head(20))